# Boulder's children against its peers: the 1990–2020 record and a cohort projection to 2050

**Brian Keegan** · *Charting Boulder*, Boulder Reporting Lab · sequel to *"Boulder's next political divide is generational"* (Oct 2025)

A descriptive comparison of the **school-age (5–17) population of the City of Boulder** against
the Boulder County ring and a basket of 25 similar U.S. cities, from the 1990–2020 decennial
censuses, rolled forward to 2050 with a **Hamilton–Perry cohort-change projection**.

**Question (H1).** Is Boulder losing children *faster* than peers facing the same national
fertility headwind — and how much of the change is fewer births versus families not staying?
School-age counts lag births by five to seventeen years, so the under-5 series is the leading
indicator and the 5–17 series the lagging one; both are reported, as level and as change.

**Claim discipline.** Descriptive only. No causal effect of land use is estimated; the land-use
covariates (zoning restrictiveness, prices, permits) are not in this repository, so the
association question ("does steeper decline travel with restrictive land use?") is out of scope
here. The projection is a **no-major-shock baseline** that rolls the 2010→2020 cohort dynamics
forward; Boulder's 2024–25 land-use reforms post-date the base period and are not in it.

**Inputs.** `data/processed/place-age-groups.csv` and `county-age-groups.csv` (built from the NHGIS
extract in `data/raw/nhgis/` by `scripts/build_age_tables.py`), `data/raw/places.csv` (the place
registry), and `data/raw/sdo/sya-county.csv` (State Demography Office county forecast, used only
to check the engine at county grain). Nothing is downloaded here.

## 1 · Setup and inputs

The 1990 and 2000 censuses report single years of age only under 20, so the finest age grain
shared by all four censuses is 19 groups (0–4 … 15–17, 18–19, 20–24 … 85+). The engine runs on the
standard 18 five-year groups (15–17 and 18–19 merged); the 5–17 band is recovered from the
observed 15–17 share of 15–19.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.options.display.max_columns = 100
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 5.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})
HIGHLIGHT, CONTEXT, ACCENT, ALT = "tab:red", "tab:gray", "tab:blue", "tab:orange"
PROC = Path("data/processed"); SDO = Path("data/raw/sdo"); OUT = Path("output"); OUT.mkdir(exist_ok=True)

CENSUS_YEARS = [1990, 2000, 2010, 2020]
PROJ_YEARS = [2030, 2040, 2050]
STEP = 10                                  # years per projection step (two 5-year groups)
GROUP_LO = list(range(0, 90, 5))           # 18 groups: 0-4 ... 80-84, 85+
G = {lo: i for i, lo in enumerate(GROUP_LO)}
MIN_DENOM = 20                             # a cohort smaller than this gives no usable ratio (tiny 1990 towns)

places = pd.read_csv("data/raw/places.csv", dtype=str)
places["tier"] = places["tier"].astype(int)
BOULDER = places.loc[places.name == "Boulder", "place_fips"].iloc[0]
PEERS = places.loc[places.tier == 2, "place_fips"].tolist()       # the similar-boulder basket
RING = places.loc[(places.tier == 1) & (places.place_fips != BOULDER), "place_fips"].tolist()
NAME = places.set_index("place_fips")["name"]
print(f"registry: Boulder + {len(RING)} ring places + {len(PEERS)} peer cities")

registry: Boulder + 5 ring places + 25 peer cities


In [2]:
def load_groups(path, idcol):
    # {fips: {year: (vec18, share_15_17)}}: 18-group vector plus the 15-17 share of 15-19
    df = pd.read_csv(path, dtype={idcol: str})
    out = {}
    for f, g in df.groupby(idcol):
        out[f] = {}
        for y, gy in g.groupby("year"):
            v = np.zeros(len(GROUP_LO))
            for lo, p in zip(gy.age_lo, gy["pop"]):
                v[G[15 if lo in (15, 18) else lo]] += p
            p1517 = gy.loc[gy.age_lo == 15, "pop"].iloc[0]; p1519 = v[G[15]]
            out[f][int(y)] = (v, p1517 / p1519 if p1519 > 0 else np.nan)
    return out

place_age = load_groups(PROC / "place-age-groups.csv", "place_fips")
county_age = load_groups(PROC / "county-age-groups.csv", "county_fips")
assert set(places.place_fips) <= set(place_age)
for f in places.place_fips:
    assert set(place_age[f]) == set(CENSUS_YEARS), f"{f} missing a census year"
totals = pd.DataFrame({NAME[f]: {y: place_age[f][y][0].sum() for y in CENSUS_YEARS} for f in places.place_fips}).T
print("loaded", len(place_age), "places and", len(county_age), "counties · total population, selected places:")
print(totals.loc[["Boulder", "Longmont", "Superior", "Madison", "Cambridge", "Provo"]].astype(int).to_string())

loaded 31 places and 28 counties · total population, selected places:
             1990    2000    2010    2020
Boulder     83312   94673   97385  108250
Longmont    51555   71093   86270   98885
Superior      255    9011   12483   13094
Madison    191262  208054  233209  269840
Cambridge   95802  101355  105162  118403
Provo       86835  105166  112488  115162


## 2 · The observed record, 1990–2020

Level *and* change, reported together: the school-age share of the population (how few children a
place has) and the change in school-age count (how fast it is falling). The 2010→2020 decade is the
peg — it is observed, not projected.

In [3]:
def band_5_17(v, s): return v[G[5]] + v[G[10]] + s * v[G[15]]
def under5(v): return v[G[0]]
def sa_share(v, s): return band_5_17(v, s) / v.sum() * 100

rows = []
for f in places.place_fips:
    r = {"place_fips": f, "name": NAME[f], "tier": int(places.set_index("place_fips").tier[f])}
    for y in CENSUS_YEARS:
        v, s = place_age[f][y]
        r[f"sa_{y}"] = band_5_17(v, s); r[f"u5_{y}"] = under5(v); r[f"share_{y}"] = sa_share(v, s)
    r["sa_chg_00_10"] = (r["sa_2010"] / r["sa_2000"] - 1) * 100
    r["sa_chg_10_20"] = (r["sa_2020"] / r["sa_2010"] - 1) * 100
    r["u5_chg_10_20"] = (r["u5_2020"] / r["u5_2010"] - 1) * 100
    rows.append(r)
obs = pd.DataFrame(rows).set_index("place_fips")
b = obs.loc[BOULDER]; peers = obs.loc[PEERS]
def pctile_below(series, value): return float((series < value).mean() * 100)
print(f"Boulder 5-17: {b.sa_2000:,.0f} (2000) -> {b.sa_2010:,.0f} (2010) -> {b.sa_2020:,.0f} (2020)"
      f"   change 2000s {b.sa_chg_00_10:+.1f}% · 2010s {b.sa_chg_10_20:+.1f}%")
print(f"Boulder under-5 2010->2020: {b.u5_chg_10_20:+.1f}%   (peer median {peers.u5_chg_10_20.median():+.1f}%; "
      f"Boulder's change is more negative than {pctile_below(-peers.u5_chg_10_20, -b.u5_chg_10_20):.0f}% of peers)")
print(f"Boulder school-age share 2020: {b.share_2020:.1f}%  (peer median {peers.share_2020.median():.1f}%; "
      f"lower than {pctile_below(-peers.share_2020, -b.share_2020):.0f}% of peers)")
print(f"peer median 5-17 change 2010->2020: {peers.sa_chg_10_20.median():+.1f}%; "
      f"Boulder's change is more negative than {pctile_below(-peers.sa_chg_10_20, -b.sa_chg_10_20):.0f}% of peers")
obs.sort_values("sa_chg_10_20")[["name", "tier", "share_2020", "sa_chg_00_10", "sa_chg_10_20", "u5_chg_10_20"]].round(1)

Boulder 5-17: 10,154 (2000) -> 9,572 (2010) -> 11,242 (2020)   change 2000s -5.7% · 2010s +17.4%
Boulder under-5 2010->2020: -17.9%   (peer median -4.8%; Boulder's change is more negative than 84% of peers)
Boulder school-age share 2020: 10.4%  (peer median 12.5%; lower than 76% of peers)
peer median 5-17 change 2010->2020: +10.8%; Boulder's change is more negative than 32% of peers


,name,tier,share_2020,sa_chg_00_10,sa_chg_10_20,u5_chg_10_20
place_fips,,,,,,
0875640,Superior,1,20.8,69.5,-11.2,-29.3
0656000,Pasadena,2,11.7,-16.2,-11.0,-22.1
4261000,Pittsburgh,2,10.2,-29.1,-10.8,-10.8
2205000,Baton Rouge,2,14.9,-9.2,-7.3,-10.8
0669070,Santa Barbara,2,12.3,-10.8,-6.1,-21.0
3673000,Syracuse,2,15.2,-12.6,-2.9,-10.6
0845970,Longmont,1,16.5,14.7,-0.1,-19.5
0473000,Tempe,2,10.6,-14.5,-0.1,4.2
2603000,Ann Arbor,2,9.7,-13.8,4.1,-0.0


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]                      # peer cities only: the ring towns grew by annexation and would swamp the scale
for f in PEERS:
    ax.plot(CENSUS_YEARS, [obs.loc[f, f"sa_{y}"] / obs.loc[f, "sa_2000"] * 100 for y in CENSUS_YEARS],
            color=CONTEXT, lw=1, alpha=0.4)
ax.plot(CENSUS_YEARS, [b[f"sa_{y}"] / b.sa_2000 * 100 for y in CENSUS_YEARS], color=HIGHLIGHT, lw=3, label="Boulder")
ax.plot(CENSUS_YEARS, [peers[f"sa_{y}"].median() / peers.sa_2000.median() * 100 for y in CENSUS_YEARS],
        color="k", lw=1.5, ls="--", label="peer median")
ax.axhline(100, color="k", lw=0.6, ls=":"); ax.set_xticks(CENSUS_YEARS)
ax.set_title("School-age (5–17) population, peer cities, index 2000 = 100"); ax.set_ylabel("Index"); ax.legend()
ax = axes[1]
ax.scatter(peers.share_2020, peers.sa_chg_10_20, color=CONTEXT, s=30, alpha=0.7, label="peer cities")
ax.scatter(obs.loc[RING].share_2020, obs.loc[RING].sa_chg_10_20, color=ACCENT, s=30, alpha=0.8, label="Boulder County ring")
ax.scatter(b.share_2020, b.sa_chg_10_20, color=HIGHLIGHT, s=90, zorder=5, label="Boulder")
for f in PEERS + RING + [BOULDER]:
    ax.annotate(NAME[f], (obs.loc[f, "share_2020"], obs.loc[f, "sa_chg_10_20"]), fontsize=7, alpha=0.8,
                xytext=(3, 2), textcoords="offset points")
ax.axhline(0, color="k", lw=0.6, ls=":")
ax.set_xlabel("5–17 share of population, 2020 (%)"); ax.set_ylabel("5–17 change 2010→2020 (%)")
ax.set_title("Level vs change (observed)"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "peers-fig1-observed-1990-2020.png", bbox_inches="tight")

## 3 · The Hamilton–Perry engine

Cohort-change ratios (CCR) carry each five-year group forward ten years: the 10–14 group in 2030
is the 0–4 group of 2020 times the ratio observed for that transition between 2010 and 2020. The
two youngest groups are generated from child ratios (0–4 per resident aged 15–44; 5–9 per resident
aged 20–49). Sexes are pooled because the 1990 table has no sex split; only the ratios matter.
The main run uses the most recent census pair (2010→2020), the standard Hamilton–Perry choice;
a sensitivity run averages the three available pairs.

In [5]:
FERT0 = slice(G[15], G[40] + 1)   # ages 15-44 -> children 0-4
FERT5 = slice(G[20], G[45] + 1)   # ages 20-49 -> children 5-9
TOP = len(GROUP_LO) - 1

def ccr_vintage(v0, v1):
    # ratios for a 10-year transition: group i at t+10 over group i-2 at t; 85+ pools 75+ at t.
    # The two child groups (indices 0, 1) are generated by child ratios, not carried forward.
    r = np.full(TOP + 1, np.nan)
    for i in range(2, TOP):
        r[i] = v1[i] / v0[i - 2] if v0[i - 2] >= MIN_DENOM else np.nan
    pool = v0[TOP - 2:].sum()
    r[TOP] = v1[TOP] / pool if pool >= MIN_DENOM else np.nan
    return r

def child_ratios(v):
    return v[G[0]] / v[FERT0].sum(), v[G[5]] / v[FERT5].sum()

def hp_project(hist, launch, vintages, n_steps, cwr=None):
    # hist: {year: vec18}; vintages: list of (t0, t1) pairs whose CCRs are averaged;
    # cwr: (r0, r5) override, else the launch-year ratios. Returns {year: vec18}.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # child indices are NaN by design
        ccr = np.nanmean([ccr_vintage(hist[t0], hist[t1]) for t0, t1 in vintages], axis=0)
    ccr[2:] = np.where(np.isnan(ccr[2:]), 1.0, ccr[2:])           # no usable vintage -> carry the cohort unchanged
    r0, r5 = cwr if cwr is not None else child_ratios(hist[launch])
    out, cur = {}, hist[launch].copy()
    for s in range(1, n_steps + 1):
        nxt = np.zeros(TOP + 1)
        for i in range(2, TOP):
            nxt[i] = cur[i - 2] * ccr[i]
        nxt[TOP] = cur[TOP - 2:].sum() * ccr[TOP]
        nxt[G[0]] = r0 * nxt[FERT0].sum()
        nxt[G[5]] = r5 * nxt[FERT5].sum()
        out[launch + s * STEP] = nxt; cur = nxt
    return out

RECENT = [(2010, 2020)]
ALL3 = [(1990, 2000), (2000, 2010), (2010, 2020)]
hist_b = {y: place_age[BOULDER][y][0] for y in CENSUS_YEARS}
demo = hp_project(hist_b, 2020, RECENT, 3)
print("engine check, Boulder 5-17 (share of 15-19 held at 2020):",
      {y: round(band_5_17(v, place_age[BOULDER][2020][1])) for y, v in demo.items()})

engine check, Boulder 5-17 (share of 15-19 held at 2020): {2030: 11058, 2040: 12063, 2050: 12532}


## 4 · Backtest: project 2020 from 2010 and compare with the census

Out-of-sample: CCRs from the 2000→2010 pair, launched from 2010, projected to 2020, scored
against the actual 2020 count. The median absolute percentage error across places becomes the
band drawn around the 2050 projection. Hamilton–Perry is validated in the literature to roughly
15 years; 2050 is 30 years out, so the band is a **floor** on uncertainty, not a confidence interval.

In [6]:
bt = []
for f in places.place_fips:
    hist = {y: place_age[f][y][0] for y in CENSUS_YEARS}; s10 = place_age[f][2010][1]; s20 = place_age[f][2020][1]
    pred = hp_project(hist, 2010, [(2000, 2010)], 1)[2020]
    sa_p, sa_a = band_5_17(pred, s10), band_5_17(hist[2020], s20)
    u5_p, u5_a = under5(pred), under5(hist[2020])
    bt.append((f, NAME[f], sa_a, sa_p, (sa_p / sa_a - 1) * 100, (u5_p / u5_a - 1) * 100))
bt = pd.DataFrame(bt, columns=["place_fips", "name", "sa_actual_2020", "sa_pred_2020", "sa_err_pct", "u5_err_pct"]).set_index("place_fips")
ENVELOPE = float(bt.loc[PEERS + [BOULDER], "sa_err_pct"].abs().median())
print(f"10-year backtest (2000-2010 ratios -> 2020), 5-17: median |error| = {ENVELOPE:.1f}% "
      f"(peers + Boulder) · mean signed error {bt.loc[PEERS, 'sa_err_pct'].mean():+.1f}%")
print(f"Boulder: predicted {bt.loc[BOULDER, 'sa_pred_2020']:,.0f} vs actual {bt.loc[BOULDER, 'sa_actual_2020']:,.0f} "
      f"({bt.loc[BOULDER, 'sa_err_pct']:+.1f}%); under-5 error {bt.loc[BOULDER, 'u5_err_pct']:+.1f}%")
bt.sort_values("sa_err_pct")[["name", "sa_actual_2020", "sa_pred_2020", "sa_err_pct", "u5_err_pct"]].round(1)

10-year backtest (2000-2010 ratios -> 2020), 5-17: median |error| = 6.4% (peers + Boulder) · mean signed error -3.9%
Boulder: predicted 9,492 vs actual 11,242 (-15.6%); under-5 error +18.4%


,name,sa_actual_2020,sa_pred_2020,sa_err_pct,u5_err_pct
place_fips,,,,,
0846355,Louisville,3780.0,2819.5,-25.4,-3.1
0841835,Lafayette,5244.0,4194.3,-20.0,-8.3
2511000,Cambridge,9207.0,7390.8,-19.7,-8.0
0807850,Boulder,11242.0,9491.6,-15.6,18.4
2603000,Ann Arbor,11983.0,10195.5,-14.9,-6.7
1938595,Iowa City,8307.0,7245.5,-12.8,-8.4
0606000,Berkeley,11297.0,9862.5,-12.7,6.9
1724582,Evanston,11041.0,9671.5,-12.4,19.3
1805860,Bloomington,7245.0,6509.3,-10.2,11.8


## 5 · Projection to 2050 and the H1 comparison

Boulder's projected 5–17 change 2020→2050 against the peer basket, with its **percentile rank**
among peers — the within-type comparison the "it's the same everywhere" counterargument cannot
absorb. The sensitivity run (three census pairs averaged) shows how much rides on the 2010s.

In [7]:
def run_all(vintages):
    proj, met = {}, []
    for f in places.place_fips:
        hist = {y: place_age[f][y][0] for y in CENSUS_YEARS}; s = place_age[f][2020][1]
        chain = {2020: hist[2020], **hp_project(hist, 2020, vintages, len(PROJ_YEARS))}
        proj[f] = chain
        sa20, sa50 = band_5_17(chain[2020], s), band_5_17(chain[2050], s)
        met.append(dict(place_fips=f, name=NAME[f], tier=int(places.set_index("place_fips").tier[f]),
                        sa_2020=sa20, sa_2030=band_5_17(chain[2030], s), sa_2040=band_5_17(chain[2040], s), sa_2050=sa50,
                        sa_chg_20_50=(sa50 / sa20 - 1) * 100, share_2020=sa20 / chain[2020].sum() * 100,
                        u5_chg_20_50=(under5(chain[2050]) / under5(chain[2020]) - 1) * 100))
    return proj, pd.DataFrame(met).set_index("place_fips")

proj, h1 = run_all(RECENT)
proj3, h1_3 = run_all(ALL3)
b1, p1 = h1.loc[BOULDER], h1.loc[PEERS]
rank_change = pctile_below(-p1.sa_chg_20_50, -b1.sa_chg_20_50)
rank_level = pctile_below(-p1.share_2020, -b1.share_2020)
print("H1 — main run (2010->2020 ratios):")
print(f"  Boulder 5-17: {b1.sa_2020:,.0f} (2020) -> {b1.sa_2030:,.0f} (2030) -> {b1.sa_2050:,.0f} (2050) = {b1.sa_chg_20_50:+.1f}%  (under-5 {b1.u5_chg_20_50:+.1f}%)")
print(f"  peer median 5-17 change 2020->2050: {p1.sa_chg_20_50.median():+.1f}%  ·  Boulder declines faster than {rank_change:.0f}% of peers")
print(f"  Boulder's 2020 school-age share is lower than {rank_level:.0f}% of peers")
b3 = h1_3.loc[BOULDER]
print(f"sensitivity (1990-2020 ratios averaged): Boulder {b3.sa_chg_20_50:+.1f}% vs peer median {h1_3.loc[PEERS].sa_chg_20_50.median():+.1f}%; "
      f"Boulder faster than {pctile_below(-h1_3.loc[PEERS].sa_chg_20_50, -b3.sa_chg_20_50):.0f}% of peers")
h1.sort_values("sa_chg_20_50")[["name", "tier", "share_2020", "sa_2020", "sa_2050", "sa_chg_20_50", "u5_chg_20_50"]].round(1)

H1 — main run (2010->2020 ratios):
  Boulder 5-17: 11,242 (2020) -> 11,058 (2030) -> 12,532 (2050) = +11.5%  (under-5 +23.9%)
  peer median 5-17 change 2020->2050: +20.0%  ·  Boulder declines faster than 56% of peers
  Boulder's 2020 school-age share is lower than 76% of peers
sensitivity (1990-2020 ratios averaged): Boulder -1.7% vs peer median +15.0%; Boulder faster than 64% of peers


,name,tier,share_2020,sa_2020,sa_2050,sa_chg_20_50,u5_chg_20_50
place_fips,,,,,,,
0656000,Pasadena,2,11.7,16236.0,10984.4,-32.3,-29.9
0669070,Santa Barbara,2,12.3,10937.0,7558.3,-30.9,-27.9
0875640,Superior,1,20.8,2725.0,1975.1,-27.5,-20.9
4261000,Pittsburgh,2,10.2,30944.0,26026.1,-15.9,-15.6
2205000,Baton Rouge,2,14.9,33853.0,28946.5,-14.5,-13.6
1724582,Evanston,2,14.1,11041.0,10098.8,-8.5,0.9
4962470,Provo,2,14.5,16714.0,15292.2,-8.5,-3.4
0952000,New Haven,2,16.1,21538.0,20491.5,-4.9,1.7
3673000,Syracuse,2,15.2,22547.0,21649.4,-4.0,-1.4


In [8]:
fig, ax = plt.subplots()
yrs = [2020, *PROJ_YEARS]
for f in PEERS:                   # peer cities only (see fig. 1); Irvine runs off the top of this scale
    m = h1.loc[f]; ax.plot(yrs, [m[f"sa_{y}"] / m.sa_2020 * 100 for y in yrs], color=CONTEXT, lw=1, alpha=0.35)
ax.plot(yrs, [p1[f"sa_{y}"].median() / p1.sa_2020.median() * 100 for y in yrs], color="k", lw=1.5, ls="--", label="peer median")
bidx = np.array([b1[f"sa_{y}"] / b1.sa_2020 * 100 for y in yrs])
ax.plot(yrs, bidx, color=HIGHLIGHT, lw=3, label="Boulder (2010→2020 ratios)", zorder=3)
ax.fill_between(yrs, bidx * (1 - ENVELOPE / 100), bidx * (1 + ENVELOPE / 100), color=HIGHLIGHT, alpha=0.12, zorder=2)
ax.plot(yrs, [b3[f"sa_{y}"] / b3.sa_2020 * 100 for y in yrs], color=HIGHLIGHT, lw=1.5, ls=":", label="Boulder (1990→2020 ratios averaged)")
ax.axhline(100, color="k", lw=0.6, ls=":"); ax.set_xticks(yrs); ax.set_ylim(50, 200)
ax.set_title(f"Projected school-age (5–17) population, peer cities, index 2020 = 100 (band = ±{ENVELOPE:.0f}% backtest error)")
ax.set_ylabel("Index"); ax.set_xlabel("Year"); ax.legend(loc="upper left", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "peers-fig2-projection-2050.png", bbox_inches="tight")

## 6 · Does the engine agree with the state? Boulder County against the SDO forecast

The State Demography Office publishes an official single-year-of-age forecast for Boulder County.
Running the same engine on the county's 1990–2020 census history and comparing it with SDO by
broad age band is the one place where an independent forecast exists to check against. Agreement
is consistency between two cohort methods, not proof; disagreement shows where the county forecast
assumes something the census trend does not contain.

In [9]:
sdo = pd.read_csv(SDO / "sya-county.csv", skiprows=1)
sdo = sdo[sdo.countyfips == 13]
def sdo_bands(y):
    s = sdo[sdo.year == y].set_index("age")["totalpopulation"]
    return {"0-4": s.loc[0:4].sum(), "5-17": s.loc[5:17].sum(), "18-24": s.loc[18:24].sum(),
            "25-64": s.loc[25:64].sum(), "65+": s.loc[65:].sum()}
def hp_bands(v, s):
    return {"0-4": v[G[0]], "5-17": band_5_17(v, s), "18-24": (1 - s) * v[G[15]] + v[G[20]],
            "25-64": v[G[25]:G[60] + 1].sum(), "65+": v[G[65]:].sum()}
ch = {y: county_age["08013"][y][0] for y in CENSUS_YEARS}; cs = county_age["08013"][2020][1]
cproj = {2020: ch[2020], **hp_project(ch, 2020, RECENT, len(PROJ_YEARS))}
cmp = pd.concat({y: pd.DataFrame({"census_or_HP": hp_bands(cproj[y], cs), "SDO": sdo_bands(y)}) for y in [2020, *PROJ_YEARS]}, axis=0)
cmp["diff_pct"] = (cmp.census_or_HP / cmp.SDO - 1) * 100
print("Boulder County: 2020 is census vs SDO estimate; 2030-2050 is Hamilton-Perry (2010->2020 ratios) vs SDO forecast")
print(cmp.round(1).to_string())
recon_5_17_2050 = float(cmp.loc[(2050, "5-17"), "diff_pct"])

Boulder County: 2020 is census vs SDO estimate; 2030-2050 is Hamilton-Perry (2010->2020 ratios) vs SDO forecast
            census_or_HP     SDO  diff_pct
2020 0-4         13858.0   13184       5.1
     5-17        48260.0   48100       0.3
     18-24       44786.0   47225      -5.2
     25-64      172591.0  171372       0.7
     65+         51263.0   51071       0.4
2030 0-4         15193.7   12526      21.3
     5-17        46306.4   38685      19.7
     18-24       46757.4   49060      -4.7
     25-64      184252.2  166744      10.5
     65+         71829.6   67099       7.1
2040 0-4         15448.3   14580       6.0
     5-17        49748.9   38270      30.0
     18-24       43022.4   43907      -2.0
     25-64      199027.7  175049      13.7
     65+         80034.1   80144      -0.1
2050 0-4         15720.7   13676      15.0
     5-17        51129.6   41372      23.6
     18-24       46140.4   42995       7.3
     25-64      211428.9  173419      21.9
     65+         83330.7   8

## 7 · Fewer births or fewer families? Two city-grain decompositions

Neither is causal, and both use the same census counts as the projection, so they corroborate
rather than independently confirm it.

1. **Implied-versus-actual children.** Apply each place's 2010 child ratios to its 2020 adults:
   the shortfall of actual 2020 children against that implied count is the part of the 2010s
   change that fertility-at-2010-rates does not explain — read as net child out-migration plus the
   change in fertility, at city grain.
2. **Frozen-fertility counterfactual.** Re-project with child ratios held at their 1990–2020 mean
   instead of the 2020 value; the gap to the main run is the share of the projected decline that
   the fall in the child ratio itself accounts for.

In [10]:
dec = []
for f in places.place_fips:
    v10, v20 = place_age[f][2010][0], place_age[f][2020][0]
    r0, r5 = child_ratios(v10)
    implied = r0 * v20[FERT0].sum() + r5 * v20[FERT5].sum()
    actual = v20[G[0]] + v20[G[5]]
    dec.append((f, NAME[f], implied, actual, (actual / implied - 1) * 100))
dec = pd.DataFrame(dec, columns=["place_fips", "name", "implied_0_9_2020", "actual_0_9_2020", "gap_pct"]).set_index("place_fips")
print("children 0-9 in 2020, actual vs implied by 2010 child ratios on 2020 adults (negative = fewer than 2010 rates imply):")
print(f"  Boulder {dec.loc[BOULDER, 'gap_pct']:+.1f}%  ·  peer median {dec.loc[PEERS, 'gap_pct'].median():+.1f}%  ·  "
      f"Boulder more negative than {pctile_below(-dec.loc[PEERS, 'gap_pct'], -dec.loc[BOULDER, 'gap_pct']):.0f}% of peers")
dec.sort_values("gap_pct")[["name", "implied_0_9_2020", "actual_0_9_2020", "gap_pct"]].round(1).head(10)

children 0-9 in 2020, actual vs implied by 2010 child ratios on 2020 adults (negative = fewer than 2010 rates imply):
  Boulder -14.7%  ·  peer median -9.7%  ·  Boulder more negative than 92% of peers


,name,implied_0_9_2020,actual_0_9_2020,gap_pct
place_fips,,,,
0845970,Longmont,14609.6,10673.0,-26.9
0875640,Superior,1899.4,1483.0,-21.9
0656000,Pasadena,14857.0,12458.0,-16.1
2754880,Rochester,18300.9,15598.0,-14.8
0807850,Boulder,8321.9,7095.0,-14.7
0827425,Fort Collins,18917.2,16232.0,-14.2
3719000,Durham,38434.1,33051.0,-14.0
4261000,Pittsburgh,29740.9,25620.0,-13.9
0824950,Erie,5473.8,4716.0,-13.8


In [11]:
ff = []
for f in places.place_fips:
    hist = {y: place_age[f][y][0] for y in CENSUS_YEARS}; s = place_age[f][2020][1]
    mean_cwr = tuple(np.mean([child_ratios(hist[y]) for y in CENSUS_YEARS], axis=0))
    frozen = hp_project(hist, 2020, RECENT, len(PROJ_YEARS), cwr=mean_cwr)
    sa20 = band_5_17(hist[2020], s); sa50 = h1.loc[f, "sa_2050"]; sa50f = band_5_17(frozen[2050], s)
    share = (sa50f - sa50) / (sa20 - sa50) * 100 if sa20 - sa50 > 0.02 * sa20 else np.nan
    ff.append((f, NAME[f], sa20, sa50, sa50f, share))
ff = pd.DataFrame(ff, columns=["place_fips", "name", "sa_2020", "sa_2050_main", "sa_2050_frozen_cwr", "fertility_share_of_decline_pct"]).set_index("place_fips")
bf = ff.loc[BOULDER]
print(f"Boulder 5-17 in 2050: {bf.sa_2050_main:,.0f} with 2020 child ratios vs {bf.sa_2050_frozen_cwr:,.0f} with 1990-2020 mean ratios")
if np.isfinite(bf.fertility_share_of_decline_pct):
    print(f"  -> the fall in Boulder's child ratio accounts for {bf.fertility_share_of_decline_pct:.0f}% of its projected 2020->2050 decline; "
          f"the rest is cohort attrition (families not staying, and fewer arriving)")
else:
    print(f"  -> the main run projects no 2020->2050 decline for Boulder, so no share is computed; the child-ratio fall alone "
          f"is worth {bf.sa_2050_frozen_cwr - bf.sa_2050_main:,.0f} school-age residents in 2050 ({(bf.sa_2050_frozen_cwr / bf.sa_2050_main - 1) * 100:+.0f}%)")
ff.dropna().sort_values("fertility_share_of_decline_pct")[["name", "sa_2020", "sa_2050_main", "sa_2050_frozen_cwr", "fertility_share_of_decline_pct"]].round(0).head(12)

Boulder 5-17 in 2050: 12,532 with 2020 child ratios vs 14,112 with 1990-2020 mean ratios
  -> the main run projects no 2020->2050 decline for Boulder, so no share is computed; the child-ratio fall alone is worth 1,580 school-age residents in 2050 (+13%)


,name,sa_2020,sa_2050_main,sa_2050_frozen_cwr,fertility_share_of_decline_pct
place_fips,,,,,
1724582,Evanston,11041.0,10099.0,10283.0,20.0
0669070,Santa Barbara,10937.0,7558.0,8256.0,21.0
0875640,Superior,2725.0,1975.0,2248.0,36.0
0656000,Pasadena,16236.0,10984.0,13902.0,56.0
2205000,Baton Rouge,33853.0,28946.0,32024.0,63.0
4962470,Provo,16714.0,15292.0,17336.0,144.0
4261000,Pittsburgh,30944.0,26026.0,36108.0,205.0
3673000,Syracuse,22547.0,21649.0,23863.0,247.0
0952000,New Haven,21538.0,20492.0,23759.0,312.0


## 8 · Pinned numbers & limits

In [12]:
def _r(x, n=0): return None if x is None or not np.isfinite(x) else (round(float(x), n) if n else int(round(float(x))))
PINNED = {
    "boulder_5_17_2010": _r(b.sa_2010), "boulder_5_17_2020": _r(b.sa_2020),
    "boulder_5_17_change_2010_2020_pct": _r(b.sa_chg_10_20, 1), "peer_median_5_17_change_2010_2020_pct": _r(peers.sa_chg_10_20.median(), 1),
    "boulder_faster_decline_than_pct_of_peers_2010_2020": _r(pctile_below(-peers.sa_chg_10_20, -b.sa_chg_10_20)),
    "boulder_under5_change_2010_2020_pct": _r(b.u5_chg_10_20, 1), "peer_median_under5_change_2010_2020_pct": _r(peers.u5_chg_10_20.median(), 1),
    "boulder_faster_under5_decline_than_pct_of_peers_2010_2020": _r(pctile_below(-peers.u5_chg_10_20, -b.u5_chg_10_20)),
    "boulder_5_17_share_2020_pct": _r(b.share_2020, 1), "boulder_share_lower_than_pct_of_peers": _r(rank_level),
    "boulder_5_17_2050_projected": _r(b1.sa_2050), "boulder_5_17_change_2020_2050_pct": _r(b1.sa_chg_20_50, 1),
    "peer_median_5_17_change_2020_2050_pct": _r(p1.sa_chg_20_50.median(), 1),
    "boulder_faster_decline_than_pct_of_peers_2020_2050": _r(rank_change),
    "boulder_5_17_change_2020_2050_pct_3vintage": _r(b3.sa_chg_20_50, 1),
    "backtest_median_abs_error_pct": _r(ENVELOPE, 1), "backtest_boulder_error_pct": _r(bt.loc[BOULDER, "sa_err_pct"], 1),
    "boulder_county_hp_vs_sdo_5_17_2050_pct": _r(recon_5_17_2050, 1),
    "boulder_children_0_9_vs_implied_2020_pct": _r(dec.loc[BOULDER, "gap_pct"], 1),
    "boulder_fertility_share_of_projected_decline_pct": _r(bf.fertility_share_of_decline_pct),
    "boulder_5_17_2050_frozen_child_ratio": _r(bf.sa_2050_frozen_cwr),
}
pinned = pd.Series(PINNED, name="value").to_frame(); pinned.to_csv(OUT / "peers-pinned.csv")
obs.round(2).to_csv(OUT / "peers-observed-1990-2020.csv"); h1.round(1).to_csv(OUT / "peers-projection-2050.csv")
bt.round(2).to_csv(OUT / "peers-backtest-2020.csv"); cmp.round(1).to_csv(OUT / "peers-boulder-county-vs-sdo.csv")
print(pinned.to_string())

                                                             value
boulder_5_17_2010                                           9572.0
boulder_5_17_2020                                          11242.0
boulder_5_17_change_2010_2020_pct                             17.4
peer_median_5_17_change_2010_2020_pct                         10.8
boulder_faster_decline_than_pct_of_peers_2010_2020            32.0
boulder_under5_change_2010_2020_pct                          -17.9
peer_median_under5_change_2010_2020_pct                       -4.8
boulder_faster_under5_decline_than_pct_of_peers_2010_2020     84.0
boulder_5_17_share_2020_pct                                   10.4
boulder_share_lower_than_pct_of_peers                         76.0
boulder_5_17_2050_projected                                12532.0
boulder_5_17_change_2020_2050_pct                             11.5
peer_median_5_17_change_2020_2050_pct                         20.0
boulder_faster_decline_than_pct_of_peers_2020_2050            

### Limits — read before citing any number

1. **Descriptive, not causal.** Nothing here identifies an effect of land use, prices, or policy on
   the child population; that comparison needs place-level covariates not in this repository.
2. **2050 is beyond the validated range.** Hamilton–Perry is validated to ~15 years; the backtest
   band is a floor on uncertainty, not a confidence interval.
3. **The 2020 census distorts college towns.** April-2020 enumeration sent students home; the
   18–24 counts feeding the 2010→2020 ratios are suspect for exactly this peer type, which also
   moves the projected 15–44 base that generates children. The three-pair sensitivity dilutes it.
4. **Annexation and boundary change** are inside the ratios: a place that grew by annexing family
   subdivisions shows cohort gains that are not in-migration to a fixed area. Boulder's growth
   boundary makes it unusually clean; Superior, Erie, and several Sunbelt peers are not.
5. **Sexes are pooled** in the child ratios (the 1990 table has no sex split). Only the ratio's
   stability matters, but a shifting sex ratio among 15–49s would bias the youngest groups.
6. **5–17 is prorated** from the 15–19 group using each place's 2020 share of 15–17 within 15–19,
   held constant. In college towns that share is small and sensitive to enrollment.
7. **The child ratios already contain net migration** of families, so the projection and the
   decompositions in §7 are not independent confirmations of each other.
8. **No county control.** Places are not scaled to a county forecast (they do not tile their
   counties, and no independent single-year county forecast exists outside Colorado). The SDO check
   in §6 is consistency at county grain, not validation of the city projection.
9. **A 2020 launch carries the 2000s birth echo forward.** The large 5–14 cohorts of 2020 become
   the projected 15–44 base that generates children, and the 2010→2020 ratios contain the families
   that arrived with them. SDO models births directly and projects a smaller Boulder County 5–17
   population than this engine does (§6); read the 2050 numbers as "if the 2010s repeat," not as
   a forecast.